# NYC Taxi Duration and Fare Regression

This notebook audits the synthetic trip data, verifies target-leakage controls, executes the model comparison, inspects residual evidence, and reconciles the selected models with exported artifacts.

## 1. Setup and data audit

Duration and fare are targets and must not be included in the predictor matrix. The benchmark is synthetic and does not represent real taxi performance.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'nyc_taxi.csv'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
df = pd.read_csv(DATA_PATH)
audit = {'shape': df.shape, 'missing_values': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'targets': ['duration', 'fare'], 'target_ranges': {column: [float(df[column].min()), float(df[column].max())] for column in ['duration', 'fare']}}
display(df.head())
audit

## 2. Execute the reproducible experiment

The command compares a mean baseline, Ridge regression, and random forest for both targets and exports model and residual diagnostics.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'run_experiment.py'), '--data', str(DATA_PATH), '--output', str(OUTPUT_DIR)], check=True)

## 3. Model comparison and visual evidence

RMSE is used for selection, while MAE and R² provide additional context. Residual plots are inspected for systematic errors.

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'model-comparison.csv')
display(comparison)
display(Image(filename=str(OUTPUT_DIR / 'model-comparison.png')))
display(Image(filename=str(OUTPUT_DIR / 'residual-diagnostics.png')))

## 4. Reconcile outputs

The JSON summary and selected-model CSV are the authoritative records for the reported results.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'results-summary.json').read_text())
selected = pd.read_csv(OUTPUT_DIR / 'selected-models.csv')
assert set(selected['target']) == {'duration', 'fare'}
summary

## Interpretation

These are estimates from synthetic data. A production model would require licensed trip data, temporal and geographic validation, traffic and weather variables, calibration, and drift monitoring.